# Volume Access Grants for ZeroBus SPN

Grants `READ_VOLUME` and `WRITE_VOLUME` on a single UC Volume to the lakeLoom
service principal.

This notebook is invoked by the `platform_bootstrap` job as a **forEach task**,
iterating over the list of managed volumes. Each iteration receives a
`volume_name` parameter identifying which volume to grant access on.

Parameters are passed from the job definition:
- `catalog_use` / `schema_use` — from `${resources.schemas.lakeloom_schema.*}`
- `spn_application_id` — from the upstream `ensure_service_principal` task via task values
- `volume_name` — from the forEach input list (e.g. `session_audio`, `screenshots`, `documents`)

In [0]:
%sql
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;

SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS active_catalog, current_schema() AS active_schema;

In [0]:
%sql
-- Tag all subsequent statements for cost tracking and audit trail.
SET QUERY_TAGS['project'] = 'lakeLoom';
SET QUERY_TAGS['component'] = 'platform_bootstrap';
SET QUERY_TAGS['pipeline'] = 'lakeloom_infra';
EXECUTE IMMEDIATE "SET QUERY_TAGS['catalog'] = '" || catalog_use || "';";
EXECUTE IMMEDIATE "SET QUERY_TAGS['schema'] = '" || schema_use || "';";

In [0]:
%sql
-- Receive spn_application_id from upstream ensure_service_principal task.
DECLARE OR REPLACE VARIABLE spn_application_id STRING;
SET VARIABLE spn_application_id = :spn_application_id;

-- Receive volume_name from the forEach task iteration.
DECLARE OR REPLACE VARIABLE volume_name STRING;
SET VARIABLE volume_name = :volume_name;

SELECT
  spn_application_id AS granting_to,
  volume_name AS target_volume,
  catalog_use || '.' || schema_use || '.' || volume_name AS fully_qualified_volume;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE read_vol_grnt_stmnt STRING DEFAULT
  'GRANT READ VOLUME ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name || ' TO `' || spn_application_id || '`;';

SELECT read_vol_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE read_vol_grnt_stmnt;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE write_vol_grnt_stmnt STRING DEFAULT
  'GRANT WRITE VOLUME ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name || ' TO `' || spn_application_id || '`;';

SELECT write_vol_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE write_vol_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE 'SHOW GRANTS ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name;

In [0]:
%sql
SELECT
  'volume_grants_applied' AS check_name,
  'read_write_granted' AS status,
  catalog_use || '.' || schema_use || '.' || volume_name AS target_volume,
  spn_application_id AS granted_to,
  current_timestamp() AS completed_at;